In [3]:
from PIL import Image
import os

In [6]:
pwd

'e:\\CEI - Carbon Stock\\experiments\\OMNI-DC\\src'

### Sentinel data with Google Earth

In [1]:
import ee

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


c:\Users\ASUS\AppData\Local\Programs\Python\Python38\lib\site-packages\google\api_core\_python_version_support.py:237: FutureWarning: You are using a non-supported Python version (3.8.2). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project='ee-khanhtran2101tq')

In [147]:
RoI = (104.64314981548942, 21.640988533288887, 105.92675553350205, 21.860378136937058)
RoI = (104.64314981548942, 21.640988533288887, 104.82675553350205, 21.860378136937058)

RoI_geo = ee.Geometry.Rectangle(RoI)
middle_point = [(RoI[0] + RoI[2]) / 2, (RoI[1] + RoI[3]) / 2]
middle_point = ee.Geometry.Point(middle_point)

region = ee.Geometry.Point([105.8342, 21.0278]).buffer(2000).bounds()

In [148]:
gedi = ee.FeatureCollection("LARSE/GEDI/GEDI02_B_002")

In [149]:
image_sen2_col = ee.ImageCollection("COPERNICUS/S2_SR") \
        .filterBounds(RoI_geo) \
        .filterDate('2023-01-01', '2023-12-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) 

lat_lon_img = ee.image.Image.pixelLonLat().clip(RoI_geo)
img_sen2 = image_sen2_col.first()
img_sen2 = img_sen2.addBands(lat_lon_img)

points = img_sen2.sample(
    region=img_sen2.geometry(),
    scale=10,          # pixel size
    geometries=True    # include lon/lat geometry
)

In [157]:
RoI = (104.64314981548942, 21.640988533288887, 104.82675553350205, 21.860378136937058)

middle_point = [(RoI[0] + RoI[2]) / 2, (RoI[1] + RoI[3]) / 2]

lat = (RoI[0] + RoI[2]) / 2
lon = (RoI[1] + RoI[3]) / 2
roi_geometry = RoI_geo

img = ee.ImageCollection('COPERNICUS/S2_SR') \
        .filterDate('2023-01-01','2023-01-10') \
        .filterBounds(ee.Geometry.Point(lon,lat)) \
        .median()
task = ee.batch.Export.image.toDrive(image=img.select(['B4','B3','B2']),
                                     description='S2_example',
                                     region=roi_geometry,
                                     folder='Sentinel',
                                     scale=50)
task.start()

In [158]:
task.status()


{'state': 'FAILED',
 'description': 'S2_example',
 'priority': 100,
 'creation_timestamp_ms': 1765441496913,
 'update_timestamp_ms': 1765441503270,
 'start_timestamp_ms': 1765441502010,
 'task_type': 'EXPORT_IMAGE',
 'attempt': 1,
 'error_message': "Image.select: Band pattern 'B4' was applied to an Image with no bands. See https://developers.google.com/earth-engine/guides/debugging#no-bands",
 'id': '6ORZP2TH2DQ5Y7P3GGOZ52SA',
 'name': 'projects/ee-khanhtran2101tq/operations/6ORZP2TH2DQ5Y7P3GGOZ52SA'}

In [137]:
lat_lon_array = lat_lon_img.toArray()

In [ ]:
lat_lon_array.sample()

AttributeError: 'Image' object has no attribute 'keys'

In [144]:
task = ee.batch.Export.image.toDrive(
    image=lat_lon_array,
    description='sentinel_array_export',
    region=lat_lon_array.geometry(),
    scale=10,
    fileNamePrefix='my_export_lyon',
    fileFormat='GEO_TIFF'
)
task.start()

In [131]:
type(points)

ee.featurecollection.FeatureCollection

In [91]:
point_data = img_sen2.sample(middle_point, 30)

In [92]:
footprint = img_sen2.geometry()
footprint_info = footprint.getInfo()

In [93]:
img_sen2

In [94]:
print("number of point", len(footprint_info['coordinates'][0]))
for coord in footprint_info['coordinates'][0]:
    lon = coord[0]
    lat = coord[1]
    print("Longitude:", lon, "Latitude:", lat)


number of point 21
Longitude: 104.0303313571821 Latitude: 22.0621309816522
Longitude: 104.03032949701284 Latitude: 22.06211263258717
Longitude: 104.03335717679204 Latitude: 21.61242750880915
Longitude: 104.0333967726887 Latitude: 21.612386208450715
Longitude: 104.03343052222024 Latitude: 21.612340691811134
Longitude: 104.03344637830217 Latitude: 21.61233799978523
Longitude: 105.09410890174044 Latitude: 21.61511963784368
Longitude: 105.09415359462072 Latitude: 21.615156215798685
Longitude: 105.0942027478824 Latitude: 21.615187238047127
Longitude: 105.09420578093352 Latitude: 21.615202034219408
Longitude: 105.09486680900217 Latitude: 22.607009475381844
Longitude: 105.09482728997513 Latitude: 22.607050940631083
Longitude: 105.09479356945847 Latitude: 22.607096638766976
Longitude: 105.09477760127974 Latitude: 22.607099497172438
Longitude: 104.16209321338759 Latitude: 22.60494070167991
Longitude: 104.16205748241127 Latitude: 22.604911261496735
Longitude: 104.1620114666014 Latitude: 22.60489

In [62]:
feat = point_data.first().getInfo()
print(feat.keys())

dict_keys(['type', 'geometry', 'id', 'properties'])


In [64]:
print(feat['properties'].keys())
print(feat['properties']['B2'])
print(feat['properties']['longitude'])
print(feat['properties']['latitude'])

dict_keys(['AOT', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'SCL', 'TCI_B', 'TCI_G', 'TCI_R', 'WVP', 'latitude', 'longitude'])
1510
104.73505930570144
21.75077379775444


In [ ]:
middle_point

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "GeometryConstructors.Point",
    "arguments": {
      "coordinates": {
        "constantValue": [
          104.73495267449573,
          21.750683335112974
        ]
      }
    }
  }
})

In [97]:
from matplotlib import pyplot as plt

plt.figure(figsize=(8, 8))
url = img_sen2.getThumbURL({
    'region': RoI_geo.buffer(2000).bounds().getInfo(),  # 1 km buffer around point
    'dimensions': 512,
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
})

# Display the thumbnail
Image(url=url)

<Figure size 576x576 with 0 Axes>

In [52]:
lat_lon_img = ee.image.Image.pixelLonLat().clip(RoI_geo)

In [57]:
print(lat_lon_img.getInfo().keys())
lat_lon_img.getInfo()['properties']

dict_keys(['type', 'bands', 'properties'])


{'system:footprint': {'type': 'Polygon',
  'coordinates': [[[104.64314981548942, 21.640988533288887],
    [104.82675553350205, 21.640988533288887],
    [104.82675553350205, 21.860378136937058],
    [104.64314981548942, 21.860378136937058],
    [104.64314981548942, 21.640988533288887]]]}}

In [58]:
lat_lon_img.getInfo()['bands']

[{'id': 'longitude',
  'data_type': {'type': 'PixelType', 'precision': 'double'},
  'dimensions': [2, 2],
  'origin': [104, 21],
  'crs': 'EPSG:4326',
  'crs_transform': [1, 0, 0, 0, 1, 0]},
 {'id': 'latitude',
  'data_type': {'type': 'PixelType', 'precision': 'double'},
  'dimensions': [2, 2],
  'origin': [104, 21],
  'crs': 'EPSG:4326',
  'crs_transform': [1, 0, 0, 0, 1, 0]}]

In [224]:
ee.Image.pixelLonLat()

In [197]:
s2_with_latlon = s2.addBands(ee.Image.pixelLonLat())

In [161]:
import ee
from IPython.display import Image

# Authenticate and initialize
ee.Authenticate()
ee.Initialize()

# Define ROI (Hanoi, Vietnam)
roi = ee.Geometry.Point([105.84117, 21.0245])
# roi = RoI_geo

# Get least cloudy Sentinel-2 collection
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .sort('CLOUDY_PIXEL_PERCENTAGE') 
# Select RGB bands
image = collection.first().select(['B4', 'B3', 'B2'])

# Visualization parameters
vis_params = {
    'min': 0,
    'max': 3000,
    'gamma': 1.3
}

# Get URL for the thumbnail
url = image.getThumbURL({
    'region': roi.buffer(1000).bounds().getInfo(),  # 1 km buffer around point
    'dimensions': 512,
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
})

# Display the thumbnail
Image(url=url)


In [162]:
task = ee.batch.Export.image.toDrive(image=image.select(['B4','B3','B2']),
                                     description='S2_example',
                                     region=roi_geometry,
                                     folder='Sentinel',
                                     scale=50)
task.start()

In [168]:
task.status()

{'state': 'COMPLETED',
 'description': 'S2_example',
 'priority': 100,
 'creation_timestamp_ms': 1765441657166,
 'update_timestamp_ms': 1765441753526,
 'start_timestamp_ms': 1765441662679,
 'task_type': 'EXPORT_IMAGE',
 'destination_uris': ['https://drive.google.com/#folders/1aFqqAiZ3kCGpKP6NzUq2xA65HEZfWLdT'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 0.15847085416316986,
 'id': 'APT52IOB2Y6TG2U2I63KA7NI',
 'name': 'projects/ee-khanhtran2101tq/operations/APT52IOB2Y6TG2U2I63KA7NI'}

In [ ]:
RoI = (104.64314981548942, 21.640988533288887, 104.82675553350205, 21.860378136937058)

middle_point = [(RoI[0] + RoI[2]) / 2, (RoI[1] + RoI[3]) / 2]

lat = (RoI[0] + RoI[2]) / 2
lon = (RoI[1] + RoI[3]) / 2
roi_geometry = RoI_geo

img = ee.ImageCollection('COPERNICUS/S2_SR') \
        .filterDate('2023-01-01','2023-01-10') \
        .filterBounds(ee.Geometry.Point(lon,lat)) \
        .median()
task = ee.batch.Export.image.toDrive(image=img.select(['B4','B3','B2']),
                                     description='S2_example',
                                     region=roi_geometry,
                                     folder='Sentinel',
                                     scale=50)
task.start()

In [180]:
import openeo

con = openeo.connect("https://openeo.cloud").authenticate_oidc()

Authenticated using refresh token.


In [181]:
RoI = [104.64314981548942, 21.640988533288887,
       104.82675553350205, 21.860378136937058]

cube = con.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={
        "west": RoI[0], "south": RoI[1],
        "east": RoI[2], "north": RoI[3]
    },
    temporal_extent=["2023-01-01", "2023-01-10"],
    bands=["B02", "B03", "B04"]
)


In [182]:
result = cube.save_result(format="GTIFF")
job = con.create_job(process_graph=result)
job.start_and_wait()
job.download_results("output_sentinel2/")

0:00:00 Job 'vito-j-25121109243743a4aa3387805b091e6f': send 'start'
0:00:12 Job 'vito-j-25121109243743a4aa3387805b091e6f': created (progress 0%)
0:00:17 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)
0:00:25 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)
0:00:34 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)
0:00:45 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)
0:00:58 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)
0:01:13 Job 'vito-j-25121109243743a4aa3387805b091e6f': queued (progress 0%)


KeyboardInterrupt: 

### Sentinel data with OpenEO

In [2]:
import openeo

# Connect to openEO back-end.
connection = openeo.connect("openeo.vito.be").authenticate_oidc()

# Load data cube from TERRASCOPE_S2_NDVI_V2 collection.
cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={"west": 3.75, "east": 4.08, "south": 51.29, "north": 51.39},
    temporal_extent=["2021-05-07", "2021-05-14"],
    bands=["B04", "B03", "B02", "B08", "B09"],
)
# Rescale digital number to physical values and take temporal maximum.

# cube.download("data_openeo/openeo_test_file_sentinelL2A_from_vito.tiff")

Authenticated using refresh token.


In [4]:
# Create a UDF object from inline source code.
udf = openeo.UDF(
"""
import xarray

def apply_datacube(cube: xarray.DataArray, context: dict) -> xarray.DataArray:
    cube.values = 0.0001 * cube.values
    lon = cube['x']
    return cube
"""
)

# Pass UDF object as child process to `apply`.
rescaled = cube.apply(process=udf)

rescaled.download("apply-udf-scaling.nc")

In [5]:
# Create a UDF object from inline source code.
udf = openeo.UDF(
"""
import xarray

def apply_datacube(cube: xarray.DataArray, context: dict) -> xarray.DataArray:
    cube.values = 0.0001 * cube.values
    lon = cube['random']
    return cube
"""
)

# Pass UDF object as child process to `apply`.
rescaled = cube.apply(process=udf)

rescaled.download("apply-udf-scaling.nc")

OpenEoApiError: [500] Internal: Server error: UDF exception while evaluating processing graph. Please check your user defined functions. stacktrace:
  File "<string>", line 6, in apply_datacube
  File "/opt/venv/lib64/python3.8/site-packages/xarray/core/dataarray.py", line 698, in __getitem__
    return self._getitem_coord(key)
  File "/opt/venv/lib64/python3.8/site-packages/xarray/core/dataarray.py", line 690, in _getitem_coord
    _, key, var = _get_virtual_variable(
  File "/opt/venv/lib64/python3.8/site-packages/xarray/core/dataset.py", line 171, in _get_virtual_variable
    ref_var = variables[ref_name]
KeyError: 'random' (ref: r-2512160936314c218f82f0d65b3e113d)

In [ ]:
rescaled_cube.download("output_openEO/rescale_s2.tiff")

In [43]:
type(cube.metadata._dimensions[3])

openeo.metadata.SpatialDimension

In [32]:
cube.metadata._dimensions[3]

SpatialDimension(type='spatial', name='y', extent=[-56, 83], crs={'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json', 'area': 'World', 'bbox': {'east_longitude': 180, 'north_latitude': 90, 'south_latitude': -90, 'west_longitude': -180}, 'coordinate_system': {'axis': [{'abbreviation': 'Lat', 'direction': 'north', 'name': 'Geodetic latitude', 'unit': 'degree'}, {'abbreviation': 'Lon', 'direction': 'east', 'name': 'Geodetic longitude', 'unit': 'degree'}], 'subtype': 'ellipsoidal'}, 'datum': {'ellipsoid': {'inverse_flattening': 298.257223563, 'name': 'WGS 84', 'semi_major_axis': 6378137}, 'name': 'World Geodetic System 1984', 'type': 'GeodeticReferenceFrame'}, 'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'}, 'name': 'AUTO 42001 (Universal Transverse Mercator)', 'type': 'GeodeticCRS'}, step=10)

In [18]:
cube.metadata.dimension_names()

['bands', 't', 'x', 'y']

In [2]:
process_ids = [process["id"] for process in connection.list_processes()]
print(process_ids[:16])

['arccos', 'arcosh', 'power', 'last', 'subtract', 'not', 'cosh', 'artanh', 'is_valid', 'first', 'median', 'eq', 'absolute', 'arctan2', 'array_labels', 'divide']


In [7]:
connection.list_processes()[0]

{'categories': ['math > trigonometric'],
 'description': 'Computes the arc cosine of `x`. The arc cosine is the inverse function of the cosine so that *`arccos(cos(x)) = x`*.\n\nWorks on radians only.\nThe no-data value `null` is passed through and therefore gets propagated.',
 'examples': [{'arguments': {'x': 1}, 'returns': 0}],
 'id': 'arccos',
 'links': [{'href': 'http://mathworld.wolfram.com/InverseCosine.html',
   'rel': 'about',
   'title': 'Inverse cosine explained by Wolfram MathWorld'}],
 'parameters': [{'description': 'A number.',
   'name': 'x',
   'schema': {'type': ['number', 'null']}}],
 'returns': {'description': 'The computed angle in radians.',
  'schema': {'type': ['number', 'null']}},
 'summary': 'Inverse cosine'}

In [6]:
cube.download("data_openeo/December15_larger.tiff")

In [8]:
cube.metadata.spatial_dimensions

[SpatialDimension(type='spatial', name='x', extent=[-180, 180], crs={'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json', 'area': 'World', 'bbox': {'east_longitude': 180, 'north_latitude': 90, 'south_latitude': -90, 'west_longitude': -180}, 'coordinate_system': {'axis': [{'abbreviation': 'Lat', 'direction': 'north', 'name': 'Geodetic latitude', 'unit': 'degree'}, {'abbreviation': 'Lon', 'direction': 'east', 'name': 'Geodetic longitude', 'unit': 'degree'}], 'subtype': 'ellipsoidal'}, 'datum': {'ellipsoid': {'inverse_flattening': 298.257223563, 'name': 'WGS 84', 'semi_major_axis': 6378137}, 'name': 'World Geodetic System 1984', 'type': 'GeodeticReferenceFrame'}, 'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'}, 'name': 'AUTO 42001 (Universal Transverse Mercator)', 'type': 'GeodeticCRS'}, step=10),
 SpatialDimension(type='spatial', name='y', extent=[-56, 83], crs={'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json', 'area': 'World', 'bbox': {'east_

In [19]:
cube_ll = cube.resample_spatial(
    # resolution=0.0001,        # ~10m (optional)
    # # projection="EPSG:4326"
    # projection="Ea6"
)


In [20]:
cube_xy = cube_ll.reduce_dimension(
    dimension="bands",
    reducer="first"
)
cube_xy.download("data_openeo/latlon_cube.tiff")

In [21]:
import tifffile

lat_lon_img = tifffile.imread("data_openeo/latlon_cube.tiff")

In [46]:
udf = openeo.UDF("""
import xarray as xr

def apply_datacube(cube: xr.DataArray, context: dict):
    lon = cube['x']
    lat = cube['y']
    return xr.Dataset({
        'longitude': lon,
        'latitude': lat
    })
""")

coords_cube = cube_ll.apply(udf)

NameError: name 'cube_ll' is not defined

In [ ]:
coords_cube.save_result(format="GTiff")

In [236]:
udf =  openeo.UDF(
"""
import xarray as xr

def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    # Extract the latitude coordinate (y)
    
    factor = context.get("factor")
    offset = context.get("offset", 0)
    return cube * factor

    # Broadcast to full cube shape (t × y × x)
    lat_grid = lat_vec.broadcast_like(cube.isel(bands=0))

    # Add a 'bands' dimension with the label 'latitude'
    lat_grid = lat_grid.expand_dims(dim={'bands': ['latitude']})

    return lat_grid
"""
)


In [ ]:
udf =  openeo.UDF(
"""
import xarray as xr

def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    # Extract the latitude coordinate (y)
    
    factor = context.get("factor")
    offset = context.get("offset", 0)
    return cube * factor

"""
)

lat = cube.apply(process=udf,
                 conext={"factor": 3.0, "offset": 0})

lat.download("data_openeo/add_sentinelL2A_from_vito.tiff")
lat_img = tifffile.imread("data_openeo/add_sentinelL2A_from_vito.tiff")

OpenEoApiError: [500] Internal: Server error: Python exception while evaluating processing graph: openeo.udf.OpenEoUdfException: No UDF found. (ref: r-25121209261147f2b8eaf6b179ef7e75)

In [ ]:
lat = cube.apply(process=udf,
                 context={"factor": 3.0, "offset": 0})

TypeError: apply() got an unexpected keyword argument 'conext'

In [ ]:
lat = cube.run_udf(process=udf)


AttributeError: 'DataCube' object has no attribute 'run_udf'

In [235]:
lat_img[:, 0, 0]

array([6864., 6744., 6752., 7044., 9269.], dtype=float32)

In [187]:
lat_img[:, 0, 0]

array([6874., 6754., 6762., 7054., 9279.], dtype=float32)

In [175]:
import xarray


def apply_datacube(cube: xarray.DataArray, context: dict) -> xarray.DataArray:
    cube.values = 0.0001 * cube.values
    return cube

In [ ]:
cube.execute_local_udf

In [90]:
mean = cube.reduce_dimension(dimension="x", reducer="mean")

In [92]:
from openeo.processes import mean

mean = cube.reduce_dimension(
    dimension="t",
    reducer=lambda data: mean(data)
)

In [96]:
scaled = cube.apply_dimension(
    dimension="bands",
    process=lambda band_values: band_values / 10000
)

In [100]:
result = cube.apply(lambda x: x + 10)
result.download("data_openeo/openeo_test_file_sentinelL2A_from_vito_added10.tiff")

In [58]:
type(result)

openeo.rest.result.SaveResult

In [28]:
processed_datacube = cube.process(
    process_id="ndvi", 
    arguments={
        "data": cube, 
        "nir": "B03", 
        "red": "B04"}
)
result = processed_datacube.save_result("GTiff")
job = result.create_job()

In [29]:
job.start_and_wait()
job.get_results().download_files("output")

0:00:00 Job 'j-2512120405004004a01764383e6fab8c': send 'start'
0:00:13 Job 'j-2512120405004004a01764383e6fab8c': queued (progress 0%)
0:00:18 Job 'j-2512120405004004a01764383e6fab8c': queued (progress 0%)
0:00:25 Job 'j-2512120405004004a01764383e6fab8c': queued (progress 0%)
0:00:33 Job 'j-2512120405004004a01764383e6fab8c': running (progress 3.9%)
0:00:43 Job 'j-2512120405004004a01764383e6fab8c': running (progress 5.4%)
0:00:56 Job 'j-2512120405004004a01764383e6fab8c': running (progress 7.3%)
0:01:11 Job 'j-2512120405004004a01764383e6fab8c': running (progress 9.5%)
0:01:31 Job 'j-2512120405004004a01764383e6fab8c': finished (progress 100%)


[WindowsPath('output/openEO_2021-05-08Z.tif'),
 WindowsPath('output/openEO_2021-05-13Z.tif'),
 WindowsPath('output/job-results.json')]

In [15]:
print(connection.list_collections())

[{'description': 'Optical data collected with airborne drone platform, and processed with MapEO Water software v1 at VITO into turbidity. Turbidity indicates the relative opacity of the water column. It is an optical water property and a measure for the amount of light scattered by constituents in the water column. The higher the scattered light intensity, the higher the turbidity. Constituents that causes water to be turbid include clay, silt, very tiny inorganic and organic matter, algae, dissolved coloured organic compounds, plankton, and other microscopic organisms. Turbidity is expressed in Formazin Nephelometric Units (FNU, according to the ISO 7027 standard).', 'extent': {'spatial': {'bbox': [[-180.0, -84.0, 180.0, 84.0]]}, 'temporal': {'interval': [['2018-02-01T00:00:00Z', None]]}}, 'id': 'MAPEO_WATER_TUR_V1', 'keywords': ['Orthoimagery', 'Water quality', 'Turbidity', 'Airborne drone data', 'UAV', 'RPAS', 'Drone', 'MicaSense', 'DJI', 'cm resolution', 'super high resultion', 'VI

In [46]:
cube.download("data_openeo/openeo_test_file_sentinelL2A_from_vito_5bands.tiff")

tiff_file = "data_openeo/openeo_test_file_sentinelL2A_from_vito_5bands.tiff"

In [ ]:
from openeo.processes import multiply
import tifffile

result = cube.apply(lambda x: multiply(x, 0.001))

result.download("data_openeo/openeo_test_file_sentinelL2A_from_vito_modified.tiff")

tiff_file_added = "data_openeo/openeo_test_file_sentinelL2A_from_vito_modified.tiff"
img_added = tifffile.imread(tiff_file_added)

In [132]:
ndvi = cube.ndvi()

# Step 2: give NDVI a "bands" dimension
ndvi = ndvi.add_dimension(name="bands", label="NDVI")

# Step 3: merge
combined = cube.merge_cubes(ndvi)

# Step 4: download
combined.download("sentinel_with_ndvi.tif")

In [139]:
added_band_img = tifffile.imread("sentinel_with_ndvi.tif")

In [144]:
print(cube.metadata.get("bands"))

None


In [113]:
ndvi_cube = cube.ndvi()
ndvi_cube.download("data_openeo/openeo_test_file_sentinelL2A_from_vito_ndvi.tiff")

ndvi_img = tifffile.imread("data_openeo/openeo_test_file_sentinelL2A_from_vito_ndvi.tiff")

In [ ]:
ndvi_cube = ndvi_cube.add_dimension(name="bands", label="NDVI")

In [114]:
ndvi_img.shape

(1141, 2313)

In [47]:
import tifffile
img = tifffile.imread(tiff_file)
print(img.shape)

(5, 1141, 2313)


In [188]:
lat_img[:, :3, 0]

array([[6874., 6874., 6890.],
       [6754., 6766., 6762.],
       [6762., 6714., 6730.],
       [7054., 7050., 7054.],
       [9279., 9279., 9279.]], dtype=float32)

In [109]:
img[:, :3, 0]

array([[6864, 6864, 6880],
       [6744, 6756, 6752],
       [6752, 6704, 6720],
       [7044, 7040, 7044],
       [9269, 9269, 9269]], dtype=uint16)

In [112]:
img_added[:, :3, 0]

array([[6.8640003, 6.8640003, 6.88     ],
       [6.7440004, 6.7560005, 6.7520003],
       [6.7520003, 6.7040005, 6.7200003],
       [7.044    , 7.0400004, 7.044    ],
       [9.269    , 9.269    , 9.269    ]], dtype=float32)

Google Earth Engine back-end

In [20]:
connection = openeo.connect("https://earthengine.openeo.org")

cube = connection.load_collection(
    "COPERNICUS/S2_HARMONIZED",
    spatial_extent={"west": 3.75, "east": 4.08, "south": 51.29, "north": 51.39},
    temporal_extent=["2021-05-07", "2021-05-14"],
    bands=["B4", "B3", "B2"],
)

# cube.download("openeo_test_file_sentinelL2A_fromGEE.tiff")

In [22]:
connection.authenticate_oidc()

OpenEoClientException: No client_id found.

##### Google Earth Engine

In [ ]:
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2023-01-01', '2023-12-31') \
    .sort('CLOUDY_PIXEL_PERCENTAGE') 

image = collection.first()
# .select(['B4', 'B3', 'B2'])

info = collection.first().getInfo()

In [254]:
image_with_latlon = image.addBands(ee.Image.pixelLonLat())
info = image_with_latlon.getInfo()

In [246]:
rgb = image.select(['B4', 'B3', 'B2'])
rgb.bandNames()

In [ ]:
lat_lon = image_with_latlon.select(['longitude', 'latitude'])

In [258]:
type(lat_lon)

ee.image.Image

In [272]:
rgb.sample(roi, scale=10).toList(10)

In [255]:
for band in info['bands']:
    print(band['id'])

B1
B2
B3
B4
B5
B6
B7
B8
B8A
B9
B11
B12
AOT
WVP
SCL
TCI_R
TCI_G
TCI_B
MSK_CLDPRB
MSK_SNWPRB
QA10
QA20
QA60
MSK_CLASSI_OPAQUE
MSK_CLASSI_CIRRUS
MSK_CLASSI_SNOW_ICE
longitude
latitude


In [214]:
value.keys()

dict_keys(['type', 'columns', 'properties', 'features'])

In [203]:
s2_with_latlon = image.addBands(ee.Image.pixelLonLat())

In [207]:
s2_with_latlon.select(['longitude', 'latitude']).getInfo()['bands']

[{'id': 'longitude',
  'data_type': {'type': 'PixelType', 'precision': 'double'},
  'crs': 'EPSG:4326',
  'crs_transform': [1, 0, 0, 0, 1, 0]},
 {'id': 'latitude',
  'data_type': {'type': 'PixelType', 'precision': 'double'},
  'crs': 'EPSG:4326',
  'crs_transform': [1, 0, 0, 0, 1, 0]}]

In [ ]:
import ee
from IPython.display import Image
from datetime import datetime

ee.Authenticate()
ee.Initialize()

# Define a polygon (example area in Hanoi)
polygon_coords = [
    [
        [105.83, 21.03],  # bottom-left
        [105.85, 21.03],  # bottom-right
        [105.85, 21.04],  # top-right
        [105.83, 21.04],  # top-left
        [105.83, 21.03]   # close polygon
    ]
]
polygon = ee.Geometry.Polygon(polygon_coords)

# Load Sentinel-2 collection and filter by polygon
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(polygon) \
    .filterDate('2023-01-01', '2023-12-31') \
    .sort('CLOUDY_PIXEL_PERCENTAGE')

# Get least cloudy image
image = collection.first()

# Print some info
time_start = image.get('system:time_start').getInfo()
date = datetime.utcfromtimestamp(time_start / 1000)
print("Acquisition date:", date)

cloud = image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
print("Cloud coverage:", cloud)

# Select RGB bands
rgb = image.select(['B4', 'B3', 'B2'])

# Generate thumbnail for the polygon
url = rgb.getThumbURL({
    'region': polygon.bounds().getInfo(),
    'dimensions': 512,
    'bands': ['B4','B3','B2'],
    'min': 0,
    'max': 3000
})

Image(url=url)  # Display thumbnail in Jupyter/Colab


### GEDI data with NASA earth access

In [1]:
import earthaccess
earthaccess.login()

In [6]:
RoI = (104.64314981548942, 21.640988533288887, 104.82675553350205, 21.860378136937058)

granules = earthaccess.search_data(
    short_name="GEDI02_B",
    temporal=("2022-01-01", "2022-01-31"),
    bounding_box=RoI
    )

filename = earthaccess.download(granules, "../gedi_L2B")

Granules found: 1
 Getting 1 granules, approx download size: 0.53 GB


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [56]:
granules = earthaccess.search_data(
    short_name="GEDI02_B",
    temporal=("2021-01-01", "2022-01-31"),
    bounding_box=RoI
    )


Granules found: 12


In [7]:
import h5py

file_path = filename[0]

with h5py.File(file_path, 'r') as f:
    print(list(f.keys()))  # List all groups
    beam = f["BEAM0001"]
    print(list(beam.keys()))
    key_name = beam.keys()
    print('rh' in key_name)
    # rh98 = f["BEAM0001"]["rh"]["rh98"][:]
    print(type(f["BEAM0001"]["geolocation"]))

    print(f["BEAM0001"]["rh"])
    print("Beam geolocation type:", type(f["BEAM0001/geolocation"]))
    print("Keys in beam geolocation:", f["BEAM0001/geolocation"].keys())

    print(f["METADATA"].keys())
    dset = f["BEAM0001/rh"]
    print("---------------")
    print(dict(f["METADATA/DatasetIdentification"].attrs))
    print("------------------")
    print(f["BEAM0001/geolocation/elev_highestreturn_a1"].attrs)
    print(dict(f["BEAM0001/geolocation/elev_highestreturn_a1"].attrs))
    # rh100 = f["BEAM0001/rh/rh100"][:]
    # lat = f["BEAM0001"]["geolocation"]["lat_lowestmode"][:]
    # lon = f["BEAM0001/geolocation/lon_lowestmode"][:]

    print(f["BEAM0001/geolocation/elev_highestreturn_a1"][:])
    print("lat_lowestmode" in f["BEAM0001/geolocation"].keys())

    print('rh98' in f['BEAM0000'].keys())
    print(f['BEAM0000'].keys())

['BEAM0000', 'BEAM0001', 'BEAM0010', 'BEAM0011', 'BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011', 'METADATA']
['algorithmrun_flag', 'ancillary', 'beam', 'channel', 'cover', 'cover_z', 'delta_time', 'fhd_normal', 'geolocation', 'l2a_quality_flag', 'l2b_quality_flag', 'land_cover_data', 'master_frac', 'master_int', 'num_detectedmodes', 'omega', 'pai', 'pai_z', 'pavd_z', 'pgap_theta', 'pgap_theta_error', 'pgap_theta_z', 'rg', 'rh100', 'rhog', 'rhog_error', 'rhov', 'rhov_error', 'rossg', 'rv', 'rx_processing', 'rx_range_highestreturn', 'rx_sample_count', 'rx_sample_start_index', 'selected_l2a_algorithm', 'selected_mode', 'selected_mode_flag', 'selected_rg_algorithm', 'sensitivity', 'shot_number', 'stale_return_flag', 'surface_flag']
False
<class 'h5py._hl.group.Group'>


KeyError: "Unable to synchronously open object (object 'rh' doesn't exist)"

In [87]:
import numpy as np

shot = gediL2B[f'{beamNames[0]}/shot_number'][()][5]  # Example shot number

index = np.where(gediL2B[f'{beamNames[0]}/shot_number'][()]==shot)[0][0]  # Set the index for the shot identified above
index

5

In [88]:
# Bring in the desired SDS
elev = gediL2B[f'{beamNames[0]}/geolocation/elev_lowestmode'][()]  # Latitude
lats = gediL2B[f'{beamNames[0]}/geolocation/lat_lowestmode'][()]  # Latitude
lons = gediL2B[f'{beamNames[0]}/geolocation/lon_lowestmode'][()]  # Longitude

shotElev = elev[index]
shotLat = lats[index]
shotLon = lons[index]
shotPAVD = pavd[index]

In [91]:
pavdAll = []
pavdElev = []

for i, e in enumerate(range(len(shotPAVD))):
    if shotPAVD[i] > 0:
        pavdElev.append((shot, shotElev + dz * i, shotPAVD[i]))  # Append tuple of shot number, elevation, and PAVD
pavdAll.append(pavdElev)                                         # Append to final list

In [ ]:
print(f"The shot is located at: {str(shotLat)}, {str(shotLon)} (shot ID: {shot}, index {index}) and is from {beamNames[0]}.")

The shot is located at: -0.30891578557757116, 92.51181903576699 (shot ID: 173040000200026003, index 5) and is from BEAM0000.


In [54]:
def print_group(name, obj):
    print(name, "→", type(obj))

with h5py.File(file_path, "r") as f:
    f.visititems(print_group)

BEAM0000 → <class 'h5py._hl.group.Group'>
BEAM0000/ancillary → <class 'h5py._hl.group.Group'>
BEAM0000/ancillary/l2a_alg_count → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/beam → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/channel → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/degrade_flag → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/delta_time → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/digital_elevation_model → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/digital_elevation_model_srtm → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/elev_highestreturn → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/elev_lowestmode → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/elevation_bias_flag → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/elevation_bin0_error → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/energy_total → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/geolocation → <class 'h5py._hl.group.Group'>
BEAM0000/geolocation/elev_highestreturn_a1 → <class 'h5py._hl.dataset.Dataset'>
BEAM0000/geolocat

### Canopy height

In [82]:
file_path = '..\\gedi_L2B\\GEDI02_B_2021148153603_O13925_03_T06072_02_003_01_V002.h5'

gediL2B = h5py.File(file_path, 'r')  # Read file using h5py
beamNames = [g for g in gediL2B.keys() if g.startswith('BEAM')]

pavd = gediL2B[f'{beamNames[0]}/pavd_z'][()]

In [128]:
file_path = '..\\gedi_L2A\\GEDI02_A_2022001020959_O17295_03_T04175_02_003_02_V002.h5'

gediL2A = h5py.File(file_path, 'r')  # Read file using h5py
beamNames = [g for g in gediL2B.keys() if g.startswith('BEAM')]

In [173]:
"latitude_1gfit" in gediL2A["BEAM0000"].keys()

False

In [172]:
gediL2A["BEAM0000/lon_highestreturn"][:]

array([ 28.54717114,  28.54799679,  28.54882212, ..., 113.09008385,
       113.09037916, 113.09067569])

In [176]:
gediL2A["BEAM0000/geolocation/latitude_1gfit"][:]

array([51.8192285 , 51.81923638, 51.81924057, ...,  0.5574875 ,
        0.55704939,  0.55667429])

In [164]:
gediL2A["BEAM0000/geolocation/shot_number"][:]

array([172950000300176621, 172950000300176622, 172950000300176623, ...,
       172950000300343803, 172950000300343804, 172950000300343805],
      dtype=uint64)

In [95]:
print(gediL2B.keys())
print(gediL2B["METADATA"].keys())
metadata = gediL2B['METADATA/DatasetIdentification']
beam0000 = gediL2B["BEAM0000"]
beam0001 = gediL2B["BEAM0001"]

<KeysViewHDF5 ['BEAM0000', 'BEAM0001', 'BEAM0010', 'BEAM0011', 'BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011', 'METADATA']>
<KeysViewHDF5 ['DatasetIdentification']>


In [140]:
beam0000.keys()

<KeysViewHDF5 ['algorithmrun_flag', 'ancillary', 'beam', 'channel', 'cover', 'cover_z', 'delta_time', 'fhd_normal', 'geolocation', 'l2a_quality_flag', 'l2b_quality_flag', 'land_cover_data', 'master_frac', 'master_int', 'num_detectedmodes', 'omega', 'pai', 'pai_z', 'pavd_z', 'pgap_theta', 'pgap_theta_error', 'pgap_theta_z', 'rg', 'rh100', 'rhog', 'rhog_error', 'rhov', 'rhov_error', 'rossg', 'rv', 'rx_processing', 'rx_range_highestreturn', 'rx_sample_count', 'rx_sample_start_index', 'selected_l2a_algorithm', 'selected_mode', 'selected_mode_flag', 'selected_rg_algorithm', 'sensitivity', 'shot_number', 'stale_return_flag', 'surface_flag']>

In [101]:
print(beam0000.keys())

<KeysViewHDF5 ['algorithmrun_flag', 'ancillary', 'beam', 'channel', 'cover', 'cover_z', 'delta_time', 'fhd_normal', 'geolocation', 'l2a_quality_flag', 'l2b_quality_flag', 'land_cover_data', 'master_frac', 'master_int', 'num_detectedmodes', 'omega', 'pai', 'pai_z', 'pavd_z', 'pgap_theta', 'pgap_theta_error', 'pgap_theta_z', 'rg', 'rh100', 'rhog', 'rhog_error', 'rhov', 'rhov_error', 'rossg', 'rv', 'rx_processing', 'rx_range_highestreturn', 'rx_sample_count', 'rx_sample_start_index', 'selected_l2a_algorithm', 'selected_mode', 'selected_mode_flag', 'selected_rg_algorithm', 'sensitivity', 'shot_number', 'stale_return_flag', 'surface_flag']>


In [125]:
beam0000['l2b_quality_flag'][:]

array([1, 1, 1, ..., 0, 0, 0], dtype=uint8)

In [119]:
import numpy as np

np.unique(beam0000['land_cover_data']["urban_proportion"])

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100], dtype=uint8)

In [100]:
beam0000['rh100'][:]

array([ 358, 1206,  498, ...,  296,  276,  253], dtype=int16)

In [99]:
beam0001['rh100'][:]

array([442, 384, 384, ..., 276, 269, 243], dtype=int16)

In [90]:
type(beam0000['rh100'])

h5py._hl.dataset.Dataset

In [79]:
"elev_lowestmode" in beam0000.keys()

False

In [59]:
metadata.attrs["abstract"]

'The GEDI L2B standard data product contains precise latitude, longitude, elevation, height, cover and vertical profile metrics for each laser footprint located on the land surface.'

In [ ]:
for field_name in metadata.attrs.keys():
    print(f"{field_name}: {metadata.attrs[field_name]}")

PGEVersion: 003
VersionID: 01
abstract: The GEDI L2B standard data product contains precise latitude, longitude, elevation, height, cover and vertical profile metrics for each laser footprint located on the land surface.
characterSet: utf8
creationDate: 2022-03-28T23:59:30.556043Z
credit: The software that generates the L2B product was implemented within the GEDI Science Data Processing System at the NASA Goddard Space Flight Center (GSFC) in Greenbelt, Maryland in collaboration with the Department of Geographical Sciences at the University of Maryland (UMD).
fileName: GEDI02_B_2022013203439_O17493_03_T10494_02_003_01_V002.h5
language: eng
originatorOrganizationName: UMD/GSFC GEDI-SDPS > GEDI Science Data Processing System
purpose: The purpose of the L2B dataset is to extract biophysical metrics from each GEDI waveform. These metrics are based on the directional gap probability profile derived from the L1B waveform and include canopy cover, Plant Area Index (PAI), Plant Area Volume Den

In [ ]:
file_path = '..\\gedi_L2B\\GEDI02_B_2021148153603_O13925_03_T06072_02_003_01_V002.h5'
gediL2B = h5py.File(file_path, 'r')  # Read file using h5py

beamNames = [g for g in gediL2B.keys() if g.startswith('BEAM')]

shotNum, dem, zElevation, zHigh, zLat, zLon, canopyHeight, quality, degrade, sensitivity, pai, beamI = ([] for i in range(12))

In [178]:
gediL2B_objs = []
gediL2B.visit(gediL2B_objs.append)                                           # Retrieve list of datasets
gediSDS = [o for o in gediL2B_objs if isinstance(gediL2B[o], h5py.Dataset)]  # Search for relevant SDS inside data file
[i for i in gediSDS if beamNames[0] in i][0:10]

gediSDS = [o for o in gediL2B_objs if isinstance(gediL2B[o], h5py.Dataset)]  # Search for relevant SDS inside data file

# Loop through each beam and open the SDS needed
for b in beamNames:
    [shotNum.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/shot_number') and b in g][0]][()]]
    [dem.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/digital_elevation_model') and b in g][0]][()]]
    [zElevation.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/elev_lowestmode') and b in g][0]][()]]  
    [zHigh.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/elev_highestreturn') and b in g][0]][()]]  
    [zLat.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/lat_lowestmode') and b in g][0]][()]]  
    [zLon.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/lon_lowestmode') and b in g][0]][()]]  
    [canopyHeight.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/rh100') and b in g][0]][()]]  
    [quality.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/l2b_quality_flag') and b in g][0]][()]]  
    [degrade.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/degrade_flag') and b in g][0]][()]]  
    [sensitivity.append(h) for h in gediL2B[[g for g in gediSDS if g.endswith('/sensitivity') and b in g][0]][()]]  
    [beamI.append(h) for h in [b] * len(gediL2B[[g for g in gediSDS if g.endswith('/shot_number') and b in g][0]][()])]  
    [pai.append(h) for h in gediL2B[f'{b}/pai'][()]]   

In [179]:
[g for g in gediSDS if g.endswith('/rh100') and b in g]

['BEAM1011/rh100']

In [180]:
gediL2B[[g for g in gediSDS if g.endswith('/rh100') and b in g][0]][()]

array([309, 318, 815, ..., 314, 302, 302], dtype=int16)

In [181]:
import pandas as pd

# Convert lists to Pandas dataframe
allDF = pd.DataFrame({'Shot Number': shotNum, 'Beam': beamI, 'Latitude': zLat, 'Longitude': zLon, 'Tandem-X DEM': dem,
                      'Elevation (m)': zElevation, 'Canopy Elevation (m)': zHigh, 'Canopy Height (rh100)': canopyHeight,
                      'Quality Flag': quality, 'Plant Area Index': pai,'Degrade Flag': degrade, 'Sensitivity': sensitivity})


In [182]:
print(len(allDF))

1337162


In [183]:
allDF.iloc[10000:10005, :]

,Shot Number,Beam,Latitude,Longitude,Tandem-X DEM,Elevation (m),Canopy Elevation (m),Canopy Height (rh100),Quality Flag,Plant Area Index,Degrade Flag,Sensitivity
10000,139250000300225933,BEAM0000,51.537334,44.997479,199.806534,200.632111,203.029617,239,0,0.048903,0,0.754472
10001,139250000300225934,BEAM0000,51.537275,44.998295,199.446793,200.494553,203.116821,261,0,0.102056,0,0.558344
10002,139250000300225935,BEAM0000,51.537215,44.999110,199.446793,200.632126,203.216934,257,0,0.108254,0,0.760224
10003,139250000300225936,BEAM0000,51.537156,44.999925,199.415146,200.626190,202.761475,213,0,0.022742,0,0.762934
10004,139250000300225937,BEAM0000,51.537096,45.000741,201.219910,200.353577,202.413940,205,0,0.065054,0,0.577861


### Draft

In [55]:
import h5py

with h5py.File(file_path, "r") as f:
    beam = f["BEAM0001"]

    print("Items in BEAM0001:", list(beam.keys()))
    print("Attributes:", dict(beam.attrs))

    for name, item in beam.items():
        if isinstance(item, h5py.Group):
            print("Group:", name, "→ contains:", list(item.keys()))
        else:
            print("Dataset:", name, "→ shape:", item.shape)


Items in BEAM0001: ['ancillary', 'beam', 'channel', 'degrade_flag', 'delta_time', 'digital_elevation_model', 'digital_elevation_model_srtm', 'elev_highestreturn', 'elev_lowestmode', 'elevation_bias_flag', 'elevation_bin0_error', 'energy_total', 'geolocation', 'land_cover_data', 'lat_highestreturn', 'lat_lowestmode', 'latitude_bin0_error', 'lon_highestreturn', 'lon_lowestmode', 'longitude_bin0_error', 'master_frac', 'master_int', 'mean_sea_surface', 'num_detectedmodes', 'quality_flag', 'rh', 'rx_1gaussfit', 'rx_assess', 'rx_processing_a1', 'rx_processing_a2', 'rx_processing_a3', 'rx_processing_a4', 'rx_processing_a5', 'rx_processing_a6', 'selected_algorithm', 'selected_mode', 'selected_mode_flag', 'sensitivity', 'shot_number', 'solar_azimuth', 'solar_elevation', 'surface_flag']
Attributes: {'description': 'Coverage beam'}
Group: ancillary → contains: ['l2a_alg_count']
Dataset: beam → shape: (167293,)
Dataset: channel → shape: (167293,)
Dataset: degrade_flag → shape: (167293,)
Dataset: d

In [21]:
key_name

ValueError: Invalid group (or file) id (invalid group (or file) ID)

In [15]:
f['/BEAM0000/shot_number']

KeyError: 'Unable to synchronously open object (invalid identifier type to function)'